In [ ]:
import math
import json
#import cv
import gc
import numpy as np
import numpy.ma as ma
import keras
from keras import backend as K
from keras.utils import np_utils
from keras.engine import Layer

from keras.layers import Input, Dense, Convolution3D, MaxPooling3D, Reshape, Flatten, BatchNormalization, Lambda, Dropout, Activation
from keras.layers.merge import concatenate
from keras.models import Model, Sequential, model_from_json
from keras.utils import multi_gpu_model
from keras.optimizers import SGD, RMSprop, Adam
from keras.layers.advanced_activations import LeakyReLU
from keras.preprocessing import image
from keras.callbacks import Callback, ModelCheckpoint, EarlyStopping

from keras.models import load_model


In [ ]:
#import vis.utils as utils
#from vis.visualization import visualize_saliency
from sklearn.preprocessing import MinMaxScaler
import sklearn.preprocessing as pre
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import tensorflow as tf
import numpy as np
import os, random, gc, pickle
os.environ["CUDA_VISIBLE_DEVICES"]="2"
import nibabel as nib
import pandas as pd
from sklearn.model_selection import train_test_split
import build_multiBranch as bm


In [ ]:
#import data_processing as process
#import build_multiBranch as bm
#import Fit_model as fitter
#import Evaluate_model as Evalu
#import generate_patches3D as generator

In [ ]:

metadata = pd.read_csv('ADNI1_Screening_1.5T.csv')
total_data = (metadata.Group !=0).values.astype('bool')

dirName = '/home/dipnilc/Dipnil/ADNI_imaging'

#listOFFiles = process.getListOfFiles(dirName)
#for elem in listOFFiles:
    #print(elem)

#print("***************")

listOfFiles = list()
for (dirpath, dirnames, filenames) in os.walk(dirName):
    listOfFiles += [os.path.join(dirpath, file) for file in filenames]


In [ ]:
metadata

In [ ]:
# make sure each subject appear at most once in the training data and so does in the test data.
subjects_count = 0
subjects_dict = []

In [ ]:
for elem in listOfFiles:
    #im = 'ADNI_002_S_0413_MR_MPR____N3__Scaled_Br_20070216232854688_S14782_I40657.nii'
    #img = nib.load(elem).get_data()
    #img = np.asarray(img)
    split_names = elem.split('/')
    subject_name = split_names[6]
    if subjects_count==0:
        subjects_count +=1
        subjects_dict.append(subject_name)
    else:
        if subject_name in subjects_dict:
            subjects_count += 0
            #subjects_dict.append(subject_name)
	    #continue
        else:
            subjects_count += 1
            subjects_dict.append(subject_name)
    if subjects_count>1072:
         break
np.shape(subjects_dict)


In [ ]:
subjects_dict1=subjects_dict

In [ ]:
np.shape(subjects_dict1)


In [ ]:
data = np.zeros((int(subjects_count), 155, 155, 95))
#data = np.zeros((int(subjects_count), 256, 256, 166))
labels = np.zeros((int(subjects_count), 1))
labels_C = np.zeros((int(subjects_count), 1))
#scaler = MinMaxScaler()

In [ ]:
idx=pd.DataFrame(np.arange(0,1073))
idx.columns=['Index']
idx

In [ ]:
metadata1=pd.concat([idx,metadata],axis=1)
metadata1[total_data]

In [ ]:
'''img = nib.load(elem).get_fdata()
img = np.asarray(img)
img = img[35:190, 35:190, 40:135]        
scaler1 = MinMaxScaler()
for jj in range(95):
    img[:, :, jj] = scaler1.fit_transform(img[:, :, jj])
    scaler2 = MinMaxScaler()
for tt in range(155):
    img[tt, :, :] = scaler2.fit_transform(img[tt, :, :])
    scaler3 = MinMaxScaler()
for kk in range(155):
    img[:, kk, :] = scaler3.fit_transform(img[:, kk, :])
#print(img)
np.shape(img)'''

In [ ]:
img = nib.load(elem).get_fdata()
np.shape(img)
#(256, 256, 166)

In [ ]:
ind_img = 0
for elem in listOfFiles:

    split_names = elem.split('/')
    subject_name = split_names[6]
    if subject_name in subjects_dict:
        img = nib.load(elem).get_fdata()
        #img = np.asarray(img)
        img = img[35:190, 35:190, 40:135]
        #img = img[0:256, 0:256, 0:166]
        scaler1 = MinMaxScaler()
        for jj in range(95):
            img[:, :, jj] = scaler1.fit_transform(img[:, :, jj])
        scaler2 = MinMaxScaler()
        for tt in range(155):
            img[tt, :, :] = scaler2.fit_transform(img[tt, :, :])
        scaler3 = MinMaxScaler()
        for kk in range(155):
            img[:, kk, :] = scaler3.fit_transform(img[:, kk, :])

        data[ind_img, :, :, :] = img
        indexM=metadata1.loc[metadata1['Subject']==(subject_name)]
    #indexM = int(indexM[0])
        group=metadata1.Group[metadata1['Subject']==(subject_name)] 
        #print(group[0])
        group=pd.DataFrame(group)       
        group = group.iloc[0]['Group']
        if group=='AD':
            label = 2
            label_c = 0
        if group=='CN':
            label = 0
            label_c = 1
        if group=='MCI':
            label = 1
            label_c = 0
        labels[ind_img] = label
        labels_C[ind_img] = label_c
        ind_img += 1
        sub_ind = subjects_dict.index(subject_name)
        subjects_dict.pop(sub_ind)


In [ ]:
listOfFiles = list()
for (dirpath, dirnames, filenames) in os.walk(dirName):
    listOfFiles += [os.path.join(dirpath, file) for file in filenames]

#make sure each subject appear at most once in the training data and so does in the test data.
subjects_count = 0
subjects_dict = []
for elem in listOfFiles:
    #im = 'ADNI_002_S_0413_MR_MPR____N3__Scaled_Br_20070216232854688_S14782_I40657.nii'
    #img = nib.load(elem).get_data()
    #img = np.asarray(img)
    split_names = elem.split('/')
    subject_name = split_names[6]
    if subjects_count==0:
        subjects_count +=1
        subjects_dict.append(subject_name)
    else:
        if subject_name in subjects_dict:
            subjects_count += 0
            #subjects_dict.append(subject_name)
	    #continue
        else:
            subjects_count += 1
            subjects_dict.append(subject_name)
    if subjects_count>999:
         break


In [ ]:
np.shape(subjects_dict)

In [ ]:
#data = scaler.fit(data)
data = np.reshape(data, (data.shape[0], data.shape[1], data.shape[2], data.shape[3], 1))
print(data.shape)


In [ ]:
X_train, X_test, Y_train, Y_test, id_train,id_test = train_test_split(data, labels,subjects_dict, test_size=0.1, train_size=0.9, random_state=7)
#X_train, X_test, Y_train_C, Y_test_C = train_test_split(data, labels_C, test_size=0.1, train_size=0.9, random_state=7)
np.save("Labels2_tr", Y_train)
np.save("Labels2_ts", Y_test)
#np.save("Labels2_C_tr", Y_train_C)
#np.save("Labels2_C_ts", Y_test_C)
Y_train = np_utils.to_categorical(Y_train, 3)
Y_test = np_utils.to_categorical(Y_test, 3)
#Y_train_C = np_utils.to_categorical(Y_train_C, 2)
#Y_test_C = np_utils.to_categorical(Y_test_C, 2)
np.save("wholeImg_tr", X_train)
np.save("wholeImg_ts", X_test)
num_tr = X_train.shape[0]
num_ts = X_test.shape[0]
print("Saved the data")

In [ ]:
num_branches = 27
batch_size = 4
nb_epochs = 10
early_stopping_patience = 20

In [ ]:
channels = 155
img_size_x = 155
img_size_y = 95
batch_size = 4
nb_classes = 3
nb_epochs = 25
learning_rate = 0.003
early_stopping_patience = 20
class_names = ["AD","CN", "MCI"]

In [ ]:
ids=pd.concat([pd.DataFrame(id_train),pd.DataFrame(id_test)],axis=0)
pd.DataFrame(ids).to_csv("/home/dipnilc/Dipnil/Codes/features_ID1.csv", header=None, sep=',',index=None)

In [ ]:
branch_input = Input(shape= (155,155,95,1))
model_temp = bm.build_separate_model(branch_input)
val_data = X_test
val_labels = Y_test
#val_labels_C = Y_test_C
#model_temp = multi_gpu_model(model_temp, gpus=2)
opt = keras.optimizers.Adam(lr=2e-4)
model_temp.compile(optimizer=opt, loss='categorical_crossentropy', metrics=[tf.keras.metrics.AUC(),'accuracy'])

v = 22071
model_weights_file = 'img_classifierW_weights_%s.h5' %v
epoch_weights_file = 'img_classifierW_weights_%s_{epoch:02d}.hdf5' %v
model_file = 'img_classifierW_model%s.h5' %v
history_file = 'img_classifierW_history_%s.json' %v


checkpoint = ModelCheckpoint(epoch_weights_file, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0, patience=early_stopping_patience, verbose=1, mode='auto')
callbacks_list = [checkpoint, early_stopping]
#datagen = ImageDataGenerator(featurewise_center=False, samplewise_center=False, featurewise_std_normalization=False,samplewise_std_normalization=False,zca_whitening=False,rotation_range=15,zoom_range=0.1,width_shift_range=0.1,height_shift_range=0.1,horizontal_flip=False,vertical_flip=False)

model_temp.summary()

In [ ]:
m = model_temp.fit(X_train, Y_train,batch_size=batch_size,epochs=100,
 verbose=1, shuffle=True, validation_data=(val_data, val_labels))


In [ ]:
def save_model_and_weights(model):
        model.save(model_file)
        model.save_weights(model_weights_file)

def save_model_history(m):
    with open(history_file, 'wb') as history_json_file:
        json.dump(m.history, history_json_file)

    return 'Saved model history to disk'
print(save_model_and_weights(model_temp))
#print(save_model_history(m))
#del model_temp
#del m
gc.collect()

In [ ]:
from keras.utils.vis_utils import plot_model
plot_model(model_temp, to_file='new_model_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
model_temp.load_weights("img_classifierW_weights_7.h5")


In [ ]:
neuron_model=Model(inputs=branch_input,outputs=model_temp.layers[-4].output)

In [ ]:
features = neuron_model.predict(X_train)

In [ ]:
features_test=neuron_model.predict(X_test)

In [ ]:
print(features.shape)
print(features_test.shape)

In [ ]:
feature_extrc=np.concatenate((features,features_test))
feature_extrc.shape

In [ ]:
feature_extrc=pd.concat([pd.DataFrame(feature_extrc),pd.DataFrame(labels)],axis=1)

In [ ]:
pd.DataFrame(feature_extrc).to_csv("/home/dipnilc/Dipnil/features_WI.csv", header=None, sep=',',index=None)

In [ ]:
df_ADNI = feature_extrc
#df_ADNI['label'] = labels
df_ADNI.head()

In [ ]:
from sklearn.decomposition import PCA
pca_ADNI = PCA(n_components=10)
#principalComponents_ADNI = pca_ADNI.fit_transform(df_ADNI[feat_cols].values)
principalComponents_ADNI = pca_ADNI.fit_transform(df_ADNI.iloc[:,:-1])

In [ ]:
np.shape(principalComponents_ADNI)
pd.DataFrame(principalComponents_ADNI).to_csv("/home/dipnilc/Dipnil/features_WI_PCA.csv", header=None, sep=',',index=None)

In [ ]:
print('Explained variation per principal component: {}'.format(pca_ADNI.explained_variance_ratio_))
variance = pca_ADNI.explained_variance_ratio_ #calculate variance ratios

var=np.cumsum(np.round(pca_ADNI.explained_variance_ratio_, decimals=3)*100)
var #cumulative sum of variance explained with [n] features

In [ ]:
import matplotlib.pyplot as plt
plt.ylabel('% Variance Explained')
plt.xlabel('# of Features')
plt.title('PCA Analysis')
plt.ylim(9,100.5)
plt.style.context('seaborn-whitegrid')


plt.plot(var)

In [ ]:
principal_ADNI_Df = pd.DataFrame(data = principalComponents_ADNI
             , columns = ['principal component 1', 'principal component 2'])
principal_ADNI_Df['y'] = labels
import matplotlib.pyplot as plt
from mlxtend.plotting import category_scatter
fig = category_scatter(x='principal component 1', y='principal component 2', label_col='y', 
                       data=principal_ADNI_Df, legend_loc='upper left')


In [ ]:
"""from keras.layers import Input, Lambda, Dense, Flatten
from keras.models import Model
from keras.applications.vgg16 import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
transfer_model=Sequential()
transfer_model.add(VGG16(include_top=False,weights='imagenet',pooling='avg'))
transfer_model.add(Dense(3,activation='softmax'))
transfer_model.layers[0].trainable=False
transfer_model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=[tf.keras.metrics.AUC(),'accuracy'])
transfer_model.summary()"""